# 04 – Embedding Chunk (Sentence-Transformer, GPU)

Masukan: `chunks_L3_S2.parquet` (notebook 02) dan `lyrecall_text.py` (notebook 01).
Keluaran: `chunk_emb_fp16.npy` (matriks N × 384, float16, sudah dinormalisasi L2) dan `embed_params.json`.

- Model: `sentence-transformers/all-MiniLM-L6-v2` (bi-encoder, vektor 384 dimensi), inferensi fp16.
- Teks yang di-*embed*: kolom `text` (huruf dan tanda baca asli, baris digabung dengan spasi), bukan `text_lex`,
  karena model dilatih pada teks wajar. Kueri harus melewati fungsi `to_dense_text` yang sama.
- Karena vektor sudah dinormalisasi, kosinus = hasil kali dalam.
- Pencarian offline: *brute force* eksak di GPU (batas atas kualitas). Indeks ANN dibandingkan terpisah pada ablasi latensi.

In [1]:
import os, sys, glob, gc, json, time
import numpy as np, pandas as pd, torch

try:
    import sentence_transformers
except ImportError:
    !pip -q install sentence-transformers
    import sentence_transformers
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/lyrecall_text.py", recursive=True)[0]))
from lyrecall_text import clean_lyrics

# ====== KONFIGURASI ======
N_LIMIT    = None
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ    = 128         # batas token per chunk
BATCH      = 512
SHARD      = 200_000     # simpan per bagian agar aman jika sesi terputus
SEED       = 42
OUT_DIR    = "/kaggle/working"

assert torch.cuda.is_available(), "Aktifkan Accelerator GPU di pengaturan notebook"
print("GPU:", torch.cuda.get_device_name(0))

def to_dense_text(t):
    """Teks yang masuk ke encoder (chunk maupun kueri): perbaiki Unicode, buang penanda, satu baris."""
    return " ".join(clean_lyrics(t).split("\n"))

GPU: Tesla T4


In [2]:
# chunk untuk model
path = glob.glob("/kaggle/input/datasets/nshamid/lyrecall-chunks/chunks_L3_S2.parquet", recursive=True)[0]
chunks = pd.read_parquet(path, columns=["chunk_id", "song_id", "text", "text_lex", "n_songs_same_text"])
if N_LIMIT:
    chunks = chunks.iloc[:N_LIMIT].reset_index(drop=True)
assert (chunks["chunk_id"].values == np.arange(len(chunks))).all()

texts   = chunks["text"].str.replace("\n", " ", regex=False).tolist()
song_of = chunks["song_id"].values
N = len(texts)
print("Chunk:", f"{N:,}")

model = SentenceTransformer(MODEL_NAME, device="cuda")
model.max_seq_length = MAX_SEQ
model.half()
print("Dimensi:", model.get_sentence_embedding_dimension(), "| max_seq_length:", model.max_seq_length)

Chunk: 1,843,304


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimensi: 384 | max_seq_length: 128


/tmp/ipykernel_22/1475026968.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensi:", model.get_sentence_embedding_dimension(), "| max_seq_length:", model.max_seq_length)


In [3]:
# cek panjang token
tok = model.tokenizer
idx = np.random.default_rng(SEED).choice(N, size=min(5000, N), replace=False)
lens = np.array([len(x) for x in tok([texts[i] for i in idx])["input_ids"]])
print("Token per chunk: median", int(np.median(lens)), "| p95", int(np.percentile(lens, 95)),
      "| p99", int(np.percentile(lens, 99)), "| maks", int(lens.max()))
print(f"> 64 token: {(lens > 64).mean():.2%} | > {MAX_SEQ} token (terpotong): {(lens > MAX_SEQ).mean():.2%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (187 > 128). Running this sequence through the model will result in indexing errors


Token per chunk: median 27 | p95 48 | p99 61 | maks 208
> 64 token: 0.82% | > 128 token (terpotong): 0.10%


## Pembuatan embedding
Per bagian (*shard*) disimpan ke disk, lalu digabung. Waktu total dicatat untuk bagian Metode (biaya fase offline).

In [4]:
def encode(txts):
    e = model.encode(txts, batch_size=BATCH, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=False)
    return e.astype(np.float16)

t0 = time.time()
parts = []
for s in tqdm(range(0, N, SHARD)):
    f = f"{OUT_DIR}/emb_shard_{s:08d}.npy"
    if os.path.exists(f):
        e = np.load(f)
    else:
        e = encode(texts[s:s + SHARD])
        np.save(f, e)
    parts.append(e)
emb = np.concatenate(parts); del parts; gc.collect()
embed_seconds = time.time() - t0

norms = np.linalg.norm(emb.astype(np.float32), axis=1)
print("Bentuk:", emb.shape, emb.dtype)
print(f"Waktu: {embed_seconds:.0f} dtk ({N / embed_seconds:,.0f} chunk/dtk)")
print("NaN:", bool(np.isnan(emb).any()), "| norma min/maks:", float(norms.min()), float(norms.max()))

  0%|          | 0/10 [00:00<?, ?it/s]

Bentuk: (1843304, 384) float16
Waktu: 327 dtk (5,636 chunk/dtk)
NaN: False | norma min/maks: 0.9995134472846985 1.0005093812942505


## Pencarian semantik (brute force di GPU) dan uji kewarasan
Skor = hasil kali dalam antara vektor kueri dan seluruh vektor chunk (kosinus, karena dinormalisasi).
Uji ini bukan evaluasi resmi; kuerinya diambil dari teks chunk itu sendiri dan dibuat sama dengan uji di notebook 03.

In [5]:
E = torch.from_numpy(emb).to("cuda")   # fp16, N x 384

@torch.no_grad()
def search_dense(q, k=100):
    qe = model.encode([to_dense_text(q)], convert_to_tensor=True, normalize_embeddings=True).to(E.dtype)
    sc = E @ qe[0]
    v, i = torch.topk(sc, min(k, N))
    return i.cpu().numpy(), v.float().cpu().numpy()

def make_variants(text, rng):
    w = text.split()
    keep = [x for x in w if rng.random() > 0.3] or w[:2]
    chars = list(text)
    for i, ch in enumerate(chars):
        if ch.isalpha() and rng.random() < 0.08:
            chars[i] = str(rng.choice(list("abcdefghijklmnopqrstuvwxyz")))
    return {"exact": text, "partial": " ".join(keep), "noisy": "".join(chars)}

def song_rank(idx, target):
    seen = []
    for c in idx:
        s = song_of[c]
        if s not in seen:
            seen.append(s)
        if len(seen) == 10:
            break
    return seen.index(target) + 1 if target in seen else None

# contoh
q = " ".join(chunks["text_lex"].iloc[123].split()[:6])
idx_, sc_ = search_dense(q, 3)
print("Kueri:", q)
for c, s in zip(idx_, sc_):
    print(f"   chunk {c} (lagu {song_of[c]}) skor {s:.3f}: {texts[c][:70]}")

# uji kewarasan
rng = np.random.default_rng(SEED)
cand = np.flatnonzero(chunks["n_songs_same_text"].values == 1)
pick = rng.choice(cand, size=min(200, len(cand)), replace=False)
res = []
for var in ["exact", "partial", "noisy"]:
    rr = []
    torch.cuda.synchronize(); t0 = time.time()
    for c in pick:
        q = make_variants(chunks["text_lex"].iloc[c], rng)[var]
        r = song_rank(search_dense(q)[0], song_of[c])
        rr.append(0.0 if r is None else 1.0 / r)
    torch.cuda.synchronize()
    res.append(("Dense (MiniLM)", var, np.mean([x > 0 for x in rr]), np.mean(rr),
                (time.time() - t0) / len(pick) * 1000))
display(pd.DataFrame(res, columns=["retriever", "varian", "Recall@10", "MRR", "ms/kueri"]).round(3))

Kueri: oh theres the cheat in the
   chunk 1440865 (lagu 78145) skor 0.613: There's no cheat codes you gotta get through it The good ones go The g
   chunk 1073213 (lagu 58278) skor 0.565: For where you are Did I have to cheat? Find the answer
   chunk 1072863 (lagu 58259) skor 0.559: But if I find out That you've been cheatin' What I don't know


,retriever,varian,Recall@10,MRR,ms/kueri
0,Dense (MiniLM),exact,0.995,0.995,13.233
1,Dense (MiniLM),partial,0.960,0.937,12.664
2,Dense (MiniLM),noisy,0.930,0.877,12.825


## Simpan
Hanya jika `N_LIMIT = None` (embedding penuh). Berkas bagian (*shard*) dihapus setelah penggabungan.

In [6]:
if N_LIMIT is None:
    np.save(f"{OUT_DIR}/chunk_emb_fp16.npy", emb)
    with open(f"{OUT_DIR}/embed_params.json", "w") as f:
        json.dump({"model": MODEL_NAME, "dim": int(emb.shape[1]), "dtype": "float16",
                   "n_chunks": int(N), "max_seq_length": MAX_SEQ, "normalized": True,
                   "text_field": "text (baris digabung spasi) -> to_dense_text",
                   "gpu": torch.cuda.get_device_name(0), "seconds": round(embed_seconds, 1),
                   "chunk_file": os.path.basename(path)}, f, indent=2)
    for fn in sorted(glob.glob(f"{OUT_DIR}/emb_shard_*.npy")):
        os.remove(fn)
    for fn in sorted(os.listdir(OUT_DIR)):
        print(f"{fn:25s} {os.path.getsize(os.path.join(OUT_DIR, fn))/1e6:9.1f} MB")
else:
    for fn in glob.glob(f"{OUT_DIR}/emb_shard_*.npy"):
        os.remove(fn)
    print("N_LIMIT masih aktif: embedding tidak disimpan.")

__notebook__.ipynb              0.0 MB
chunk_emb_fp16.npy           1415.7 MB
embed_params.json               0.0 MB
